<a href="https://colab.research.google.com/github/hjiwoong/bigdata/blob/main/selenium%2B.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip install selenium webdriver-manager beautifulsoup4 pandas openpyxl -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 28.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 510.3/510.3 kB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.6/131.6 kB 9.3 MB/s eta 0:00:00


In [ ]:
from selenium import webdriver #브라우저를 실행하고 제어
from selenium.webdriver.chrome.service import Service # 크롬드라이버 실행 파일의 경로와 실행 방식을 관리
from selenium.webdriver.chrome.options import Options # 브라우저 옵션(설정)을 지정
from selenium.webdriver.common.by import By #HTML 요소를 찾는 방법을 지정 By.ID / By.CSS_SELECTOR / By.XPATH 등
from selenium.webdriver.support.ui import WebDriverWait # 특정 조건이 만족될 때까지 대기
from selenium.webdriver.support import expected_conditions as EC #요소가 HTML에 존재할 때까지 대기
from webdriver_manager.chrome import ChromeDriverManager # 현재 크롬 버전에 맞는 크롬드라이버를 자동으로 다운로드
from bs4 import BeautifulSoup # HTML 문자열을 파싱해서 원하는 태그와 텍스트를 쉽게 찾아주는 라이브러리
import pandas as pd
import time #지정한 초만큼 실행을 멈추고 대기


크롬 드라이버 실행

In [ ]:

chrome_options = Options()
chrome_options.add_experimental_option("detach", True)# detach=True : 코드 실행이 끝나도 브라우저가 자동으로 닫히지 않습니다
chrome_options.add_experimental_option("excludeSwitches", ["enable-logging"])# 크롬 실행 시 출력되는 불필요한 로그 메시지를 숨깁니다

service = Service(ChromeDriverManager().install())#알맞은 버전의 크롬드라이버를 자동 설치하고 경로를 반환
#반환된 경로로 드라이버 실행 서비스 객체를 만듭니다
driver = webdriver.Chrome(service=service, options=chrome_options) #실제 크롬 브라우저를 실행
print("드라이버 실행 완료") #이후 모든 조작은 이 driver 객체로 합니다

WebDriverException: Message: unknown error: cannot find Chrome binary
Stacktrace:
#0 0x58b961c9b4e3 <unknown>
#1 0x58b9619cac76 <unknown>
#2 0x58b9619f1757 <unknown>
#3 0x58b9619f0029 <unknown>
#4 0x58b961a2eccc <unknown>
#5 0x58b961a2e47f <unknown>
#6 0x58b961a25de3 <unknown>
#7 0x58b9619fb2dd <unknown>
#8 0x58b9619fc34e <unknown>
#9 0x58b961c5b3e4 <unknown>
#10 0x58b961c5f3d7 <unknown>
#11 0x58b961c69b20 <unknown>
#12 0x58b961c60023 <unknown>
#13 0x58b961c2e1aa <unknown>
#14 0x58b961c846b8 <unknown>
#15 0x58b961c84847 <unknown>
#16 0x58b961c94243 <unknown>
#17 0x7de84b025ac3 <unknown>


페이지 접속/ 상품 링크 10개 수집

In [ ]:
main_url = "https://www.oliveyoung.co.kr/store/main/getBestList.do?dispCatNo=900000100100001&fltDispCatNo=10000010001&pageIdx=1&rowsPerPage=8" #크롤링할 페이지 URL을 변수에 저장

driver.get(main_url) # 브라우저가 해당 URL로 이동

wait = WebDriverWait(driver, 400) #봇 접근 시 캡차를 띄우므로 수동으로 풀 수 있게 최대 300초(5분) 동안 대기

wait.until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, "div.best-area"))) #요소가 페이지에 나타날 때까지 기다려라
time.sleep(2) #요소 감지 후 2초 추가 대기,페이지 내 다른 요소들까지 완전히 렌더링될 시간

soup = BeautifulSoup(driver.page_source, "html.parser") #현재 브라우저에 렌더링된 HTML 전체를 문자열로 가져와서 파싱하여 soup 객체를 만듬
items = soup.select("#Container > div.best-area > div.TabsConts.on > ul > li")#조건에 맞는 요소를 모두 찾아 리스트로 반환
sub_url = [item.select_one("div.prd_info a.prd_thumb")["href"] for item in items[:10]] # 리스트 컴프리헨션으로 각 상품 카드에서 링크(href)를 뽑아 sub_url 리스트를 만듭니다

print(f"수집된 링크: {len(sub_url)}개")

for i, url in enumerate(sub_url, 1):
    print(f"{i}, {url.split('goodsNo=')[1].split('&')[0]}")

상품 상세 페이지 접속 & 상품 정보 수집

In [ ]:
result = [] # 수집 결과를 담을 빈 리스트

for i, url in enumerate(sub_url, 1):
    print(f"[{i}/10] 크롤링 중...", end=" ")

    driver.get(url) # 브라우저가 상품 상세 페이지로 이동
    time.sleep(3) #JavaScript가 상품 정보를 렌더링할 시간을 확보
    soup = BeautifulSoup(driver.page_source, "html.parser") #상세페이지 html을 파싱

    # 브랜드명
    try:
        brand = soup.select_one('[class*="btn-brand"]').get_text(strip=True)
    except:
        brand = "없음"

    # 상품명
    try:
        name = soup.select_one('[class*="GoodsDetailInfo_title"]').get_text(strip=True)
    except:
        name = "없음"

    # 카테고리
    try:
        breacrumb = soup.select_one('[class*="Breadcrumb_breadcrumb-inner"]')
        links = breacrumb.select('a[role="link"]') # 리스트 컴프리헨션 + join() : ["스킨케어","크림","크림"] → "스킨케어 > 크림 > 크림"
        category = ">".join([a.get_text(strip=True)  for a in links])
    except:
        category = "없음"
    # 정가
    try:
        price = soup.select_one('[data-qa-name="text-product-original-price"] span:first-child').get_text(strip=True) + "원"
    except:
        price = "없음"

    # 할인가
    try:
        discount = soup.select_one('[data-qa-name="text-product-discount-price"] span:first-child').get_text(strip=True) + "원"
    except:
        discount = "없음"

    # 평점
    try:
        rating_tag = soup.select_one('span.rating') # select_one() : 조건에 맞는 요소를 하나만 반환
        for blind in rating_tag.select('.oyblind'): # select() : 조건에 맞는 요소를 모두 찾아 리스트로 반환
            blind.decompose() # decompose()로 제거
        rating = rating_tag.get_text(strip=True)
    except:
        rating = '없음'

    # 리뷰건수
    try:
        review_count = soup.select_one('div[class*="review-count"] button span').get_text(strip=True) + "건"
    except:
        review_count = "없음"

    result.append([i, brand, name, category, price, discount, rating, review_count])

    print(f"{brand} / {name[:25]} / 평점 {rating} / 리뷰 {review_count}")

driver.quit() # 모든 수집이 끝나면 브라우저를 종료
print(f"\n 수집 완료 - 총 {len(result)}개")


데이터프레임 확인

In [ ]:
columns = ["순위", "브랜드", "상품명", "카테고리", "정가", "할인가", "평점", "리뷰건수"]
df = pd.DataFrame(result, columns=columns)
df

엑셀 파일 저장

In [ ]:
df.to_excel("올리브영_랭킹TOP10.xlsx", index=False)
print("엑셀 저장 완료 — 올리브영_랭킹TOP10.xlsx")